# Lesson 12.2 - ML Pipelines, Deployment & Monitoring (toy pipeline demo)

This notebook demonstrates a compact production-style pipeline:

- train + validate,
- generate inference slices,
- compute drift metrics,
- trigger alerts using operational rules.

## Objectives

1. Build a reproducible training + inference flow.
2. Compare reference and production-like feature distributions.
3. Trigger monitoring alerts from simple rules.

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 7
np.random.seed(SEED)

## Build Training and Reference Data

In [2]:
X, y = make_classification(
    n_samples=3000,
    n_features=12,
    n_informative=7,
    n_redundant=3,
    weights=[0.75, 0.25],
    random_state=SEED,
)

feature_cols = [f"f{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_cols)
df["target"] = y

train_df, holdout_df = train_test_split(df, test_size=0.3, stratify=df["target"], random_state=SEED)

X_train = train_df[feature_cols]
y_train = train_df["target"]
X_holdout = holdout_df[feature_cols]
y_holdout = holdout_df["target"]

## Train Model and Validate

In [3]:
model = GradientBoostingClassifier(random_state=SEED)
model.fit(X_train, y_train)

holdout_proba = model.predict_proba(X_holdout)[:, 1]
holdout_pred = (holdout_proba >= 0.5).astype(int)

validation_metrics = {
    "holdout_accuracy": float(accuracy_score(y_holdout, holdout_pred)),
    "holdout_auc": float(roc_auc_score(y_holdout, holdout_proba)),
}
validation_metrics

{'holdout_accuracy': 0.8911111111111111, 'holdout_auc': 0.9173141499368336}

## Simulate Production Batch with Drift

In [4]:
prod_df = holdout_df.copy()
# Inject mild drift in a subset of features.
prod_df["f0"] = prod_df["f0"] * 1.25 + 0.4
prod_df["f3"] = prod_df["f3"] - 0.3

X_prod = prod_df[feature_cols]
prod_proba = model.predict_proba(X_prod)[:, 1]
prod_pred = (prod_proba >= 0.5).astype(int)

prod_metrics = {
    "prod_positive_rate": float(prod_pred.mean()),
    "prod_avg_score": float(prod_proba.mean()),
}
prod_metrics

{'prod_positive_rate': 0.17444444444444446,
 'prod_avg_score': 0.23026525201933917}

## Drift Metric: Population Stability Index (PSI)

In [5]:
def psi(expected: pd.Series, actual: pd.Series, bins: int = 10) -> float:
    eps = 1e-6
    quantiles = np.linspace(0, 1, bins + 1)
    cuts = np.quantile(expected, quantiles)
    cuts[0], cuts[-1] = -np.inf, np.inf

    exp_counts, _ = np.histogram(expected, bins=cuts)
    act_counts, _ = np.histogram(actual, bins=cuts)

    exp_dist = exp_counts / max(exp_counts.sum(), 1)
    act_dist = act_counts / max(act_counts.sum(), 1)

    exp_dist = np.clip(exp_dist, eps, None)
    act_dist = np.clip(act_dist, eps, None)

    return float(np.sum((act_dist - exp_dist) * np.log(act_dist / exp_dist)))


psi_scores = {
    col: psi(train_df[col], prod_df[col], bins=10)
    for col in feature_cols
}

psi_df = pd.DataFrame({"feature": list(psi_scores.keys()), "psi": list(psi_scores.values())}).sort_values("psi", ascending=False)
psi_df.head(6)

,feature,psi
0,f0,0.108951
3,f3,0.046922
6,f6,0.022284
4,f4,0.019869
11,f11,0.016601
8,f8,0.012345


## Alerting Rules

In [6]:
ALERT_RULES = {
    "psi_warn": 0.15,
    "psi_critical": 0.25,
    "auc_floor": 0.82,
}

alerts = []

if validation_metrics["holdout_auc"] < ALERT_RULES["auc_floor"]:
    alerts.append("CRITICAL: baseline AUC below release floor")

for _, row in psi_df.iterrows():
    if row["psi"] >= ALERT_RULES["psi_critical"]:
        alerts.append(f"CRITICAL: severe drift detected in {row['feature']} (PSI={row['psi']:.3f})")
    elif row["psi"] >= ALERT_RULES["psi_warn"]:
        alerts.append(f"WARN: moderate drift detected in {row['feature']} (PSI={row['psi']:.3f})")

alerts[:8] if alerts else ["No alerts"]

['No alerts']

## Production Case Studies & Exceptions

### Case 1: Fraud model with sudden geo expansion
- New customer geography changed feature distributions.
- PSI-based drift alerts triggered retraining before KPI collapse.

### Case 2: Recommendation model in seasonal business
- Periodic seasonality looked like drift but was expected.
- Team used seasonality-aware thresholds to reduce false alarms.

### Case 3 (Exception): Stable industrial process
- Extremely stable data made frequent retraining unnecessary.
- Monitoring remained active, but retraining cadence was intentionally sparse.

## Interview Questions & Answers

1. **Q: What does an ML pipeline include?**
   **A:** Data prep, training, validation, artifact handling, deployment handoff, and monitoring.

2. **Q: Why monitor drift after deployment?**
   **A:** Input data can change and silently degrade model quality.

3. **Q: What is PSI used for?**
   **A:** To quantify distribution shift between reference and current data.

4. **Q: How do you avoid alert fatigue?**
   **A:** Use severity tiers, persistence checks, and feature-level ownership.

5. **Q: What is a warning vs critical drift threshold?**
   **A:** Warning indicates investigation needed; critical often triggers immediate mitigation.

6. **Q: When should you retrain immediately?**
   **A:** When drift + KPI degradation indicate material business risk.

7. **Q: Why can drift exist without immediate KPI drop?**
   **A:** Some shifts affect non-critical regions of the feature space first.

8. **Q: How do batch and online monitoring differ?**
   **A:** Batch uses delayed aggregates; online needs low-latency metrics and alerts.

9. **Q: What is deployment safety gate?**
   **A:** A check that blocks rollout if quality/reliability thresholds fail.

10. **Q: How do you validate a monitoring system itself?**
   **A:** Backtest alerts on historical incidents and synthetic perturbations.
